In [ ]:
import numpy as np
import cv2
from pathlib import Path
import csv

DEPTH_DIR = Path("/Users/devrathod/Documents/classes_2026/Senior Design/Eco-Cars-Depth-Estimation-2026/data/monodepth_v2/depth_arrays")
CSV_FILE = "/Users/devrathod/Documents/classes_2026/Senior Design/Eco-Cars-Depth-Estimation-2026/data/monodepth_v2/depth_analysis_results.csv"

quantize_depth = True
round_digits = 2

files = sorted(DEPTH_DIR.glob("frame_*_depth.npy"))

print(f"Found {len(files)} depth frames")

# Load first frame to get size
sample = np.load(files[0])
H, W = sample.shape
total_pixels = H * W

results = []

for f in files:

    frame_index = int(f.stem.split("_")[1])

    depth = np.load(f)

    # Quantize depth (optional)
    if quantize_depth:
        depth_q = np.round(depth, round_digits)
    else:
        depth_q = depth

    # Valid pixels
    valid_mask = depth > 0
    valid_pixels = np.sum(valid_mask)

    percent_valid = (valid_pixels / total_pixels) * 100

    # Connected components
    depth_binary = valid_mask.astype(np.uint8)
    num_labels, _ = cv2.connectedComponents(depth_binary)
    connected_objects = num_labels - 1

    # Unique depth values
    unique_depth_values = len(np.unique(depth_q[valid_mask]))

    stats = {
        "frame_index": frame_index,
        "valid_pixels": int(valid_pixels),
        "percent_valid": round(float(percent_valid), 2),
        "connected_objects": int(connected_objects),
        "unique_depth_values": int(unique_depth_values)
    }

    results.append(stats)

# Print results
for r in results:
    print(
        f"Frame {r['frame_index']} | "
        f"valid_pixels={r['valid_pixels']} | "
        f"percent_valid={r['percent_valid']}% | "
        f"connected_objects={r['connected_objects']} | "
        f"unique_depth_values={r['unique_depth_values']}"
    )

# Save CSV
with open(CSV_FILE, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=results[0].keys())
    writer.writeheader()
    writer.writerows(results)

print(f"\nResults saved to {CSV_FILE}")

Saved to depth_statistics.npy
[{'frame_index': 0, 'valid_pixels': 4882, 'percent_valid': 0.19864908854166666, 'connected_objects': 4537, 'unique_depth_values': 4882}, {'frame_index': 1, 'valid_pixels': 4851, 'percent_valid': 0.19738769531250003, 'connected_objects': 4510, 'unique_depth_values': 4850}, {'frame_index': 2, 'valid_pixels': 4803, 'percent_valid': 0.19543457031250003, 'connected_objects': 4456, 'unique_depth_values': 4802}, {'frame_index': 3, 'valid_pixels': 4762, 'percent_valid': 0.19376627604166666, 'connected_objects': 4411, 'unique_depth_values': 4760}, {'frame_index': 4, 'valid_pixels': 4892, 'percent_valid': 0.19905598958333334, 'connected_objects': 4538, 'unique_depth_values': 4891}]
